# Temmar et al. (2025), LINK: multi-day training and fine-tuning (Figure 5)

Paper: doi:10.52202/085713-4642. Dataset: DANDI:001201, version 0.261026.1912, streamed (the most recent
published version; the paper cites none, and the three published versions hold the same assets, see
`signals.ipynb`). Claims file: `claims/temmar-2025.yaml`. Code read: github.com/chesteklab/LINK_dataset at
5e1c36a.

| Claim | Here |
|---|---|
| fig5a-multiday-short-term | tested on 12 day-0 sessions, 1 and 5 training sessions |
| fig5a-long-term-decay | tested on the same models |
| fig5b-finetune | blocked, too large (the estimate is computed below) |

Why a subset. Each LSTM takes about 195 s to train when four train at once on this runner
(`decoding.ipynb` prints the times). The paper trains 5 LSTMs on each of 283 day-0 sessions, 1415 fits,
about 19 hours here. The criteria of both Figure 5A claims compare only 1 and 5 training sessions, so
only those two are trained, on 12 day-0 sessions spread evenly over the eligible ones, chosen before
any result was seen: 24 fits.

In [ ]:
import os, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from scipy.optimize import curve_fit
import link

T0 = time.time()
ITERS = int(os.environ.get("LINK_DEV_ITERS", link.LSTM_ITERS))   # 2000 (paper); lowered only while developing
N_DAY0 = int(os.environ.get("LINK_DEV_NDAY0", "12"))              # 12 day-0 sessions (see above)
EVAL_EVERY = 5      # score every 5th test bin (cost)
HORIZON = 180       # paper, Section 4.3: evaluated on all sessions within 180 days of day 0
A = link.assets()
counts = A.groupby("date").size()
single = A[A["date"].map(counts) == 1].reset_index(drop=True)      # paper, Section 4.3: days with two sessions excluded
eligible = single.index[10:-1]   # and the first 10 sessions; the last, with no later session to test on, is left out
print(f"{len(A)} sessions; on single-session days {len(single)}; eligible day-0 sessions {len(eligible)} "
      f"(the Figure 5A legend reports 283 days)")
day0_idx = eligible[np.linspace(0, len(eligible) - 1, N_DAY0).round().astype(int)]
print("day-0 sessions:", [single["date"][i].date().isoformat() for i in day0_idx])

## Load the sessions needed

For each day 0, its training sessions (the latest 1 or 5 single-session days up to and including day 0)
and every single-session day within 180 days after it.

In [ ]:
need = set()
for i in day0_idx:
    need.update(range(i - 4, i + 1))
    d0 = single["date"][i]
    need.update(single.index[(single["date"] >= d0) & (single["date"] <= d0 + pd.Timedelta(days=HORIZON))])
need = sorted(need)


def load(i):
    s = link.read_session(single["asset_id"][i], tcr=False)
    (xtr, ytr), (xte, yte) = link.split(s)
    return i, {"date": single["date"][i], "xtr": xtr, "ytr": ytr, "xte": xte, "yte": yte}


t = time.time()
with ThreadPoolExecutor(4) as ex:
    SES = dict(ex.map(load, need))
print(f"loaded {len(SES)} sessions in {time.time() - t:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
L3 = pd.DataFrame([{"date": v["date"], "train_bins": len(v["xtr"]), "test_bins": len(v["xte"])} for v in SES.values()])
print("sessions loaded:", len(L3), "| by year:", L3["date"].dt.year.value_counts().sort_index().to_dict())
print("training bins %d to %d, test bins %d to %d" % (L3["train_bins"].min(), L3["train_bins"].max(), L3["test_bins"].min(), L3["test_bins"].max()))
print("SBP (stored units): %.3f to %.3f" % (min(v["xte"].min() for v in SES.values()), max(v["xte"].max() for v in SES.values())))
yk = np.vstack([v["yte"] for v in SES.values()])
for n, a, b in zip(link.KIN_NAMES, yk.min(0), yk.max(0)):
    print(f"  hold-out {n}: {a:.3f} to {b:.3f}")

## Train and evaluate

The LSTM and its training are those of `decoding.ipynb` (Appendix C.1). For k training sessions the
training bins of the k sessions are z-scored together (one mean and SD per channel), the windows never
cross a session boundary, and the model sees 2000 batches in all. Every model is then evaluated on the
hold-out trials of each session within 180 days after day 0, with input normalisation held at the
training parameters (Appendix C.1, "For the analyses in Figure 5").

In [ ]:
def task(arg):
    i, k = arg
    t0 = time.time()
    tr = [SES[j] for j in range(i - k + 1, i + 1)]
    mu, sd = link.zparams(np.concatenate([s["xtr"] for s in tr]))
    segs = [(((s["xtr"] - mu) / sd).astype(np.float32), s["ytr"]) for s in tr]
    state, onorm = link.lstm_train(segs, seed=42, iters=ITERS)
    fit_s = time.time() - t0
    d0 = SES[i]["date"]
    rows = []
    for j, s in SES.items():
        rel = (s["date"] - d0).days
        if j < i or rel > HORIZON:
            continue
        p = link.lstm_predict(state, onorm, ((s["xte"] - mu) / sd).astype(np.float32), every=EVAL_EVERY)
        rows.append({"day0": d0, "k": k, "test": s["date"], "rel": rel,
                     "r2": link.r2_per_output(s["yte"][::EVAL_EVERY], p).mean()})
    return {"rows": rows, "fit_s": fit_s, "span_days": (d0 - tr[0]["date"]).days, "k": k, "day0": d0}


t = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
    RES = list(ex.map(task, [(i, k) for i in day0_idx for k in (1, 5)]))
fits = np.array([r["fit_s"] for r in RES])
print(f"{len(RES)} fits in {(time.time() - t) / 60:.1f} min; seconds per fit {fits.min():.0f} to {fits.max():.0f}")
MD = pd.DataFrame([row for r in RES for row in r["rows"]])
spans = [r["span_days"] for r in RES if r["k"] == 5]
print("days spanned by the 5 training sessions: %d to %d" % (min(spans), max(spans)))
print("evaluations per model: %d to %d" % (MD.groupby(["day0", "k"]).size().min(), MD.groupby(["day0", "k"]).size().max()))

## fig5a-multiday-short-term

> Training the LSTM on up to five consecutive past sessions reduces the initial drop in performance within
> about 5 days of training and improves performance over time, but slightly lowers day-0 performance.

In [ ]:
curve = MD.groupby(["k", "rel"])["r2"].mean().unstack(0)
rows = []
for k in (1, 5):
    c = curve[k].dropna()
    d0 = c.get(0, np.nan)
    near = c.loc[1:5].mean()
    rows.append({"k": k, "day0 R2": d0, "mean R2 days 1-5": near, "drop": d0 - near,
                 "mean days 6-30": c.loc[6:30].mean(), "mean days 31-180": c.loc[31:180].mean(),
                 "relative days with data 1-5": int(c.loc[1:5].notna().sum())})
print(pd.DataFrame(rows).round(4).to_string(index=False))
per = MD.pivot_table(index=["day0", "test"], columns="k", values="r2")
per["rel"] = [(t - d).days for d, t in per.index]
for lab, sel in (("day 0", per["rel"] == 0), ("days 1-5", per["rel"].between(1, 5)),
                 ("days 6-180", per["rel"].between(6, 180))):
    d = per[sel]
    print(f"paired, {lab}: {len(d)} evaluations, 5 sessions higher in {(d[5] > d[1]).mean():.3f}, "
          f"mean difference {(d[5] - d[1]).mean():+.4f}")
print("per day 0 (day-0 R2, k=1 / k=5):")
print(per[per["rel"] == 0][[1, 5]].round(3).to_string())

**Result: reproduced, on 12 day-0 sessions.** Against one training session, five raise the mean R2 over relative days 1 to 5 from -0.502 to -0.251, shrink the drop from day 0 to those days from 1.089 to 0.791, and lower day-0 R2 from 0.587 to 0.540 (lower on 11 of 12 day-0 sessions). Paired over the 16 evaluations at 1 to 5 days, five sessions are better in 81% of them. The R2 values after day 0 are negative because the input normalisation is held at the training day's, as Appendix C.1 says for Figure 5.

## fig5a-long-term-decay

> Double-exponential fits to the single-session and five-session conditions over 0 to 180 days show steeper
> decay for single-session training; the initial decay decreases with more training sessions, but the
> long-term decay is not significantly affected.

In [ ]:
def dexp(x, a, b, c, d):
    return a * np.exp(b * x) + c * np.exp(d * x)


fitres = {}
for k in (1, 5):
    c = curve[k].dropna()
    try:
        p, cov = curve_fit(dexp, c.index.to_numpy(float), c.to_numpy(), p0=(0.374, 0.00365, 0.134, 0.140), maxfev=10000)
        se = np.sqrt(np.diag(cov))
        fast, slow = (1, 3) if abs(p[1]) > abs(p[3]) else (3, 1)
        res = c.to_numpy() - dexp(c.index.to_numpy(float), *p)
        fitres[k] = {"params": p, "se": se, "fast": fast, "slow": slow, "drop5": dexp(0, *p) - dexp(5, *p),
                     "rmse": float(np.sqrt(np.mean(res ** 2)))}
        print(f"k={k}: a={p[0]:.4f} b={p[1]:.4f} c={p[2]:.4f} d={p[3]:.5f}; initial rate {p[fast]:.4f} "
              f"(95% {p[fast] - 1.96 * se[fast]:.4f} to {p[fast] + 1.96 * se[fast]:.4f}); long-term rate {p[slow]:.5f} "
              f"(95% {p[slow] - 1.96 * se[slow]:.5f} to {p[slow] + 1.96 * se[slow]:.5f}); predicted drop day 0 to 5 "
              f"{fitres[k]['drop5']:.4f}; residual RMS {fitres[k]['rmse']:.4f}; points {len(c)}")
    except Exception as e:
        print(f"k={k}: fit failed: {e}")
if len(fitres) == 2:
    f1, f5 = fitres[1], fitres[5]
    lo = lambda f, j: f["params"][j] - 1.96 * f["se"][j]
    hi = lambda f, j: f["params"][j] + 1.96 * f["se"][j]
    overlap = not (hi(f1, f1["slow"]) < lo(f5, f5["slow"]) or hi(f5, f5["slow"]) < lo(f1, f1["slow"]))
    print("initial rate more negative for k=1:", f1["params"][f1["fast"]] < f5["params"][f5["fast"]],
          "| larger predicted drop to day 5 for k=1:", f1["drop5"] > f5["drop5"],
          "| long-term rate intervals overlap:", overlap,
          "| long-term rate difference (k=1 minus k=5): %.5f" % (f1["params"][f1["slow"]] - f5["params"][f5["slow"]]))
fig, ax = plt.subplots(figsize=(7, 3))
for k in (1, 5):
    c = curve[k].dropna()
    ax.plot(c.index, c, ".", ms=3, label=f"{k} session(s)")
    if k in fitres:
        xx = np.arange(0, HORIZON + 1)
        ax.plot(xx, dexp(xx, *fitres[k]["params"]))
ax.set_xlabel("days after day 0"); ax.set_ylabel("R2"); ax.legend(); plt.show()
print("plotted mean R2: %.3f to %.3f" % (np.nanmin(curve.to_numpy()), np.nanmax(curve.to_numpy())))

**Result: not reproduced.** Fit to the mean R2 per relative day from 12 day-0 sessions, the double exponentials are degenerate. For one session the fit has a growing component of near-zero amplitude and a single negative-amplitude component with rate -0.0029 per day. For five sessions the fit has two growing components of opposing amplitude. Neither condition holds: the initial rate is not more negative for one session and the predicted drop from day 0 to day 5 is not larger (-0.009 against 0.014). The long-term rates' intervals overlap, but they are very wide (five sessions: -1.48 to 1.50 per day). The residual RMS is 0.40 and 0.65 R2 on a curve that ranges down to -3.98, so 12 day-0 sessions are likely too few for this fit; the raw short-term comparison above does show the smaller drop with five sessions.

## fig5b-finetune: the size of the computation

> Continual daily fine-tuning of an LSTM on 30 s to 5 min of each new session's data, over up to 200 days
> from 20 seed days, reduces both short- and long-term decay, and with 120 s or 300 s later decoders
> outperform the day-0 decode.

Not run. The cell estimates its cost from the seed days of `continual_learning_lstm.py` (STARTING_DAYS,
named in the claim's criterion), the sessions within 200 days after each (CO on two-session days), the
fine-tuning iterations per duration of the criterion (50, 250, 250, 500), the two normalisation readings,
and the seconds per iteration measured above.

In [ ]:
STARTING_DAYS = ["2020-07-08", "2020-10-02", "2020-12-12", "2021-02-27", "2021-04-05", "2021-05-05", "2021-06-16",
                 "2021-07-09", "2021-07-29", "2021-08-19", "2021-09-14", "2021-10-27", "2022-01-17", "2022-02-21",
                 "2022-04-01", "2022-04-21", "2022-06-01", "2022-08-05", "2022-09-01", "2022-09-30"]
days = pd.Series(sorted(A["date"].unique()))
present = [d for d in STARTING_DAYS if pd.Timestamp(d) in set(days)]
n_later = [int(((days > pd.Timestamp(d)) & (days <= pd.Timestamp(d) + pd.Timedelta(days=200))).sum()) for d in STARTING_DAYS]
sec_per_iter = np.median(fits) / ITERS
iters = len(STARTING_DAYS) * 2000 + sum(n_later) * (50 + 250 + 250 + 500) * 2
print(f"seed days in the dataset: {len(present)} of {len(STARTING_DAYS)}; later sessions within 200 days: "
      f"{min(n_later)} to {max(n_later)} per seed, {sum(n_later)} in all")
print(f"training iterations: {iters}; at {sec_per_iter:.3f} s per iteration with 4 fits at once: "
      f"{iters * sec_per_iter / 4 / 3600:.1f} h, without the evaluations")

**Result: blocked, too large.** The fine-tuning runs 20 seed days, each followed by 32 to 85 sessions within 200 days, four durations and two normalisation readings: 2,446,600 training iterations, about 16.5 hours at the measured 0.097 s per iteration with four fits at once, beyond the notebook limit and the run's time. It was not run.

## Summary

| Claim | Outcome |
|---|---|
| fig5a-multiday-short-term | reproduced (12 day-0 sessions) |
| fig5a-long-term-decay | not reproduced |
| fig5b-finetune | blocked, too large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: 12 of the 283 day-0 sessions, and only 1 and 5 training sessions, for time; scored on every 5th test bin; the code's variant of the multi-day trainer (90/10 split, refit normalisation, no noise) was not run.

What was found about the dataset: dropping the 9 two-session days, the first 10 sessions and the last session (which has no later session) leaves 283 day-0 sessions, the count in the Figure 5A legend.

In [ ]:
print(f"notebook time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")